# EA2 – Ideación, diseño y documentación de la necesidad

**Proyecto Integrador III** · Braian Rico · Samuel Rosero · Diego Cuervo

En este notebook obtenemos un dataset de Hugging Face por código, lo cargamos en un `DataFrame` de pandas, hacemos una revisión descriptiva y generamos automáticamente un resumen `.txt` en `EA2/resultados/`.

## 1. Necesidad analítica (escenario hipotético)

> **Aviso:** el escenario es **hipotético**, planteado con fines académicos. No existe un cliente real.

El equipo de operaciones de un sistema público de bicicletas compartidas necesita saber **cómo cambia la demanda por hora** según la franja horaria, el tipo de día (laboral, no laboral o festivo), la estación del año y el clima, y **en qué se diferencian los usuarios casuales de los registrados**. Con eso podría planificar la redistribución de bicicletas y el mantenimiento.

**Pregunta descriptiva:** ¿Cómo varía la cantidad de alquileres por hora según la franja horaria, el tipo de día, la estación y las condiciones del clima, y qué diferencias hay entre usuarios casuales y registrados durante 2011 y 2012?

**Unidad de análisis:** cada fila representa **una hora de servicio** del sistema, con la cantidad de alquileres de esa hora y las condiciones de calendario y clima.

## 2. Dataset elegido

| Dato | Valor |
|---|---|
| Dataset en Hugging Face | [`t22000t/bike-sharing-tabular`](https://huggingface.co/datasets/t22000t/bike-sharing-tabular) |
| Split | `train` |
| Revisión (commit) | `6b645e0cedbc5208fcca11ac94544b088478bf61` |
| Fuente original | [Bike Sharing Dataset – UCI Machine Learning Repository (id 275)](https://archive.ics.uci.edu/dataset/275/bike+sharing+dataset) |
| Autores originales | Hadi Fanaee-T y João Gama |
| Licencia de la fuente | CC BY 4.0 |
| Fecha de consulta | 2026-10-25 |

El dataset de Hugging Face es una copia tabular del archivo **por hora** del dataset de UCI: datos del sistema *Capital Bikeshare* (Washington D.C.) entre 2011 y 2012. Fijamos la **revisión** para que cualquiera obtenga exactamente los mismos datos, aunque el repositorio cambie después.

## 3. Preparación del entorno

Buscamos la raíz del repositorio (la carpeta que contiene `src/clases_po`) y agregamos `src` a la ruta de Python para poder importar nuestras clases.

In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
while ROOT != ROOT.parent and not (ROOT / "src" / "clases_po").exists():
    ROOT = ROOT.parent

sys.path.insert(0, str(ROOT / "src"))

import pandas as pd

from clases_po import HFDatasetLoader, Reportes

/home/runner/work/Proyecto-integrado-III/Proyecto-integrado-III/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 4. Obtención del dataset y carga en un DataFrame

Usamos la clase `HFDatasetLoader`. Cargamos el dataset **completo** (sin `max_rows` y sin *streaming*).

In [2]:
DATASET = "t22000t/bike-sharing-tabular"
CONFIG = None
SPLIT = "train"
REVISION = "6b645e0cedbc5208fcca11ac94544b088478bf61"

loader = HFDatasetLoader(
    dataset_name=DATASET,
    config=CONFIG,
    split=SPLIT,
    revision=REVISION,
)
df = loader.load()
loader.info()

Generating train split: 0 examples [00:00, ? examples/s]

Generating train split: 17379 examples [00:00, 831350.47 examples/s]

{'dataset': 't22000t/bike-sharing-tabular',
 'config': None,
 'split': 'train',
 'revision': '6b645e0cedbc5208fcca11ac94544b088478bf61',
 'filas': 17379,
 'columnas': 17}

## 5. Revisión descriptiva del DataFrame

### 5.1 Algunos registros

In [3]:
df.head(10)

,instant,dteday,season,yr,mnth,hr,holiday,weekday,workingday,weathersit,temp,atemp,hum,windspeed,casual,registered,cnt
0,1,2011-01-01,1,0,1,0,0,6,0,1,0.24,0.2879,0.81,0.0000,3,13,16
1,2,2011-01-01,1,0,1,1,0,6,0,1,0.22,0.2727,0.80,0.0000,8,32,40
2,3,2011-01-01,1,0,1,2,0,6,0,1,0.22,0.2727,0.80,0.0000,5,27,32
3,4,2011-01-01,1,0,1,3,0,6,0,1,0.24,0.2879,0.75,0.0000,3,10,13
4,5,2011-01-01,1,0,1,4,0,6,0,1,0.24,0.2879,0.75,0.0000,0,1,1
5,6,2011-01-01,1,0,1,5,0,6,0,2,0.24,0.2576,0.75,0.0896,0,1,1
6,7,2011-01-01,1,0,1,6,0,6,0,1,0.22,0.2727,0.80,0.0000,2,0,2
7,8,2011-01-01,1,0,1,7,0,6,0,1,0.20,0.2576,0.86,0.0000,1,2,3
8,9,2011-01-01,1,0,1,8,0,6,0,1,0.24,0.2879,0.75,0.0000,1,7,8
9,10,2011-01-01,1,0,1,9,0,6,0,1,0.32,0.3485,0.76,0.0000,8,6,14


### 5.2 Filas y columnas

In [4]:
filas, columnas = df.shape
print(f"Filas: {filas:,}".replace(",", "."))
print(f"Columnas: {columnas}")

Filas: 17.379
Columnas: 17


### 5.3 Nombres de columnas y tipos de dato

In [5]:
df.dtypes.rename("tipo").to_frame()

,tipo
instant,int64
dteday,object
season,int64
yr,int64
mnth,int64
hr,int64
holiday,int64
weekday,int64
workingday,int64
weathersit,int64


### 5.4 Valores nulos por columna

In [6]:
df.isna().sum().rename("nulos").to_frame()

,nulos
instant,0
dteday,0
season,0
yr,0
mnth,0
hr,0
holiday,0
weekday,0
workingday,0
weathersit,0


### 5.5 Estadísticas básicas de las columnas numéricas

In [7]:
df.describe().round(3).T

,count,mean,std,min,25%,50%,75%,max
instant,17379.0,8690.000,5017.029,1.00,4345.500,8690.000,13034.500,17379.000
season,17379.0,2.502,1.107,1.00,2.000,3.000,3.000,4.000
yr,17379.0,0.503,0.500,0.00,0.000,1.000,1.000,1.000
mnth,17379.0,6.538,3.439,1.00,4.000,7.000,10.000,12.000
hr,17379.0,11.547,6.914,0.00,6.000,12.000,18.000,23.000
holiday,17379.0,0.029,0.167,0.00,0.000,0.000,0.000,1.000
weekday,17379.0,3.004,2.006,0.00,1.000,3.000,5.000,6.000
workingday,17379.0,0.683,0.465,0.00,0.000,1.000,1.000,1.000
weathersit,17379.0,1.425,0.639,1.00,1.000,1.000,2.000,4.000
temp,17379.0,0.497,0.193,0.02,0.340,0.500,0.660,1.000


## 6. ¿Qué representa cada fila y qué variables sirven para el análisis?

Cada fila es **una hora de servicio** del sistema entre el 1 de enero de 2011 y el 31 de diciembre de 2012.

### Diccionario de variables

Basado **solo** en la documentación oficial de UCI (id 275), consultada el 2026-10-25. Lo que la documentación no define se verificó contra las fechas del propio dataset o se marca como *pendiente de verificar*.

| Variable | Significado | Tipo | Unidad / valores |
|---|---|---|---|
| `instant` | Índice del registro | Entero (identificador) | Contador |
| `dteday` | Fecha | Fecha | Fecha calendario |
| `season` | Estación del año | Categórica | 1 = invierno, 2 = primavera, 3 = verano, 4 = otoño |
| `yr` | Año | Categórica | 0 = 2011, 1 = 2012 |
| `mnth` | Mes | Categórica | 1 a 12 |
| `hr` | Hora del día | Categórica | 0 a 23 |
| `holiday` | Indica si el día es festivo (según el calendario de feriados de Washington D.C.) | Binaria | 1 = festivo, 0 = no festivo (la documentación no detalla la codificación; verificado: los días con 1 en 2011 son festivos del calendario de Washington D. C., p. ej. 4 de julio y 15 de abril) |
| `weekday` | Día de la semana | Categórica | 0 = domingo a 6 = sábado (la documentación no lo indica; verificado contra `dteday`, p. ej. 2011-01-01 fue sábado y tiene 6) |
| `workingday` | Día laboral | Binaria | 1 si el día no es fin de semana ni festivo; 0 en otro caso |
| `weathersit` | Situación del clima | Categórica | 1 = despejado o pocas nubes; 2 = niebla con nubes; 3 = nieve o lluvia ligera (con tormenta o nubes dispersas); 4 = lluvia fuerte, granizo, tormenta o nieve con niebla |
| `temp` | Temperatura normalizada | Real | Celsius normalizado: (t − t_min)/(t_max − t_min), con t_min = −8 y t_max = +39 |
| `atemp` | Sensación térmica normalizada | Real | Celsius normalizado con t_min = −16 y t_max = +50 |
| `hum` | Humedad normalizada | Real | Valor dividido por 100 (máximo) |
| `windspeed` | Velocidad del viento normalizada | Real | Valor dividido por 67 (máximo); unidad original *pendiente de verificar* |
| `casual` | Alquileres de usuarios casuales | Entero (conteo) | Bicicletas alquiladas en esa hora |
| `registered` | Alquileres de usuarios registrados | Entero (conteo) | Bicicletas alquiladas en esa hora |
| `cnt` | Total de alquileres | Entero (conteo) | `casual` + `registered` |

### Variables que apoyan la pregunta

- **Demanda (lo que queremos describir):** `cnt`, `casual` y `registered`.
- **Franja horaria:** `hr`.
- **Tipo de día:** `workingday`, `holiday` y `weekday`.
- **Estación:** `season` (apoyada por `mnth`).
- **Clima:** `weathersit`, `temp`, `atemp`, `hum` y `windspeed`.
- **Periodo 2011–2012:** `yr` y `dteday`.

`instant` solo identifica el registro y no aporta al análisis.

> En esta entrega **no se transforma** el dataset: solo se revisa de forma descriptiva.

## 7. Resumen `.txt` generado desde el DataFrame

Con la clase `Reportes` creamos `EA2/resultados/resumen_dataframe.txt` (UTF-8, ruta relativa al notebook). El encabezado indica dataset, split, configuración, revisión y si se usó una muestra. La carpeta se crea si no existe y el archivo se regenera en cada ejecución.

In [8]:
RUTA_RESUMEN = Path("../resultados/resumen_dataframe.txt")

# La carga fue completa si no se limitó max_rows ni se usó streaming
es_muestra = loader.max_rows is not None or loader.streaming
n_filas = f"{len(df):,}".replace(",", ".")
if es_muestra:
    muestra = f"sí, {n_filas} filas"
else:
    muestra = f"no, dataset completo ({n_filas} filas)"

encabezado = {
    "Dataset": loader.dataset_name,
    "Configuración": loader.config or "por defecto (sin configuración)",
    "Split": loader.split,
    "Revisión": loader.revision or "no fijada",
    "Muestra": muestra,
}

ruta = Reportes().guardar_reporte(df, RUTA_RESUMEN, encabezado=encabezado)
print(f"Archivo generado: {ruta}")

Archivo generado: ../resultados/resumen_dataframe.txt


## 8. Verificación: leemos el archivo y lo imprimimos

Volvemos a abrir el `.txt` desde el disco para comprobar que su contenido coincide con el DataFrame.

In [9]:
contenido = RUTA_RESUMEN.read_text(encoding="utf-8")
print(contenido)

REPORTE DEL DATAFRAME
Dataset: t22000t/bike-sharing-tabular
Configuración: por defecto (sin configuración)
Split: train
Revisión: 6b645e0cedbc5208fcca11ac94544b088478bf61
Muestra: no, dataset completo (17.379 filas)
Filas: 17379
Columnas: 17
Duplicados: 0

Columnas y tipos:
- instant: int64
- dteday: object
- season: int64
- yr: int64
- mnth: int64
- hr: int64
- holiday: int64
- weekday: int64
- workingday: int64
- weathersit: int64
- temp: float64
- atemp: float64
- hum: float64
- windspeed: float64
- casual: int64
- registered: int64
- cnt: int64

Valores nulos por columna:
- instant: 0
- dteday: 0
- season: 0
- yr: 0
- mnth: 0
- hr: 0
- holiday: 0
- weekday: 0
- workingday: 0
- weathersit: 0
- temp: 0
- atemp: 0
- hum: 0
- windspeed: 0
- casual: 0
- registered: 0
- cnt: 0

Estadísticas numéricas:
         instant     season         yr       mnth         hr    holiday    weekday  workingday  weathersit       temp      atemp        hum  windspeed     casual  registered        cnt
coun

In [10]:
# Comprobaciones automáticas contra el DataFrame
assert f"Filas: {len(df)}" in contenido
assert f"Columnas: {len(df.columns)}" in contenido
assert all(columna in contenido for columna in df.columns)
print("El contenido del .txt coincide con el DataFrame.")

El contenido del .txt coincide con el DataFrame.


## 9. Conclusión

- Se obtuvo por código el dataset `t22000t/bike-sharing-tabular` (split `train`, revisión fija) y se cargó completo en un DataFrame.
- El DataFrame tiene una fila por hora de servicio y variables de demanda, calendario y clima que permiten responder la pregunta descriptiva.
- Se generó automáticamente `EA2/resultados/resumen_dataframe.txt` y se verificó leyéndolo de nuevo.
- La codificación de `weekday` y `holiday`, que UCI no documenta, se verificó contra las fechas del dataset. Queda pendiente de verificar la unidad original de `windspeed`.